In [ ]:
import pandas as pd

# 1. 데이터 로드 (메모리 절약을 위해 필요한 컬럼만 로드)
prior = pd.read_csv('../data/raw/order_products__prior.csv', usecols=['order_id', 'product_id', 'add_to_cart_order', 'reordered'])
orders = pd.read_csv('../data/raw/orders.csv', usecols=['order_id', 'user_id'])

# 2. User_id 연결 (Merge)
prior = pd.merge(prior, orders, on='order_id', how='left')

# --- [A] 상품별 특징 (Product-level Features) ---
product_features = prior.groupby('product_id').agg({
    'reordered': 'mean',
    'add_to_cart_order': 'mean',
    'order_id': 'count'
}).rename(columns={
    'reordered': 'prod_reorder_rate',
    'add_to_cart_order': 'prod_avg_cart_pos',
    'order_id': 'prod_total_sales'
}).reset_index()

# --- [B] 유저-상품 간 관계 특징 (User-Product Features) ---
user_product_features = prior.groupby(['user_id', 'product_id']).size().reset_index(name='user_prod_count')

# 3. 최종 결과물 저장
product_features.to_csv('product_features.csv', index=False)
print("상품별 특징 표(product_features.csv) 생성 완료!")